In [160]:
from comet_ml import API
import pandas as pd
import re

api_key = "rsWnsJtGaRo3twpTYJlHagKLm"
workspace = "kirill-korolev"
project_name = "sampling-bench"

api = API(api_key=api_key)
experiments = api.get_experiments(workspace=workspace, project_name=project_name)


In [161]:
def filter_experiments(experiments, filter_name):
    filtered_experiments = []
    for experiment in experiments:
        if filter_name in experiment.name:
            filtered_experiments.append(experiment)
    return filtered_experiments

def get_target_sd(experiment):
    for metric in experiment.get_metrics():
        if metric["metricName"] == "discrepancies/sd_target":
            return float(metric["metricValue"])

def get_metrics(experiment, steps):
    metric_name_to_table_name = {"KL/eubo": "EUBO", "KL/elbo": "ELBO", "logZ/delta_reverse": "$\Delta \log Z$", "Z/delta_reverse": "$\Delta Z$", "discrepancies/sd": "SD"}
    result = {}
    for metric in experiment.get_metrics():
        if metric["metricName"] not in metric_name_to_table_name.keys():
            continue
        if metric["step"] in steps:
            continue
        result[metric_name_to_table_name[metric["metricName"]]] = metric["metricValue"]
    algorithm_name = [param["valueCurrent"] for param in experiment.get_parameters_summary() if param["name"] == "algorithm_name"][0]
    algorithm_name = algorithm_name.replace("_", "\_")
    result["algorithm_name"] = "$\\text{" + algorithm_name + "}$"
    return result

def create_latex_table(experiments, filter_name):
    # Collect all metric dicts with algorithm name as one entry
    rows = []
    for experiment in experiments:
        row = get_metrics(experiment, [19999, 20000])
        rows.append(row)

    target_sd = get_target_sd(experiments[0])

    df = pd.DataFrame(rows)
    df = df.set_index("algorithm_name").rename_axis("Algorithm")
    df = df.replace("NaN", "-").fillna("-")
    df = df[sorted(df.columns)]
    df = df.sort_index()

    for col in df.columns:
        df[col] = df[col].apply(lambda x: round(float(x), 3) if isinstance(x, (float, int)) or (isinstance(x, str) and x.replace('.', '', 1).replace('-', '', 1).isdigit()) else x)

    def sci_to_tex(val):
        # Only attempt conversion for floats or convertible strings
        try:
            num = float(val)
        except Exception:
            return val
        s = f"{num:.3f}"
        # Check if value is in scientific notation
        if "e" in s or "E" in s:
            coeff, exp = f"{num:.3e}".split("e")
            coeff = coeff.rstrip('0').rstrip('.') if '.' in coeff else coeff
            exp = int(exp)
            return f"{coeff} \\times 10^{{{exp}}}"
        return s

    latex_table = df.to_latex(
        escape=False,
        float_format=sci_to_tex,
        label=f"table:{filter_name}",
        caption=None,
    )

    # Make sure that any values like 3.145e3 (output directly as text by pandas) are fixed in the table as well
    def replace_scientific(match):
        val = match.group(0)
        return sci_to_tex(val)

    latex_table = re.sub(
        r"(-?\d+\.\d+e[+-]?\d+|-?\d+e[+-]?\d+)",
        replace_scientific,
        latex_table,
    )

    for param in experiments[0].get_parameters_summary():
        if param["name"] == "algorithm_name":
            algorithm_name = param["valueCurrent"]
            env_name = experiments[0].name.split(f"{algorithm_name}_", 1)[-1] if f"{algorithm_name}_" in experiments[0].name else ""
            break
    env_name = env_name.replace("_", "\_")
    caption_text = f"{env_name}. Target SD {target_sd:.3f}."
    caption_text = "\\caption{" + caption_text + "}\n"
    latex_table = latex_table.replace("\\begin{table}", "\\begin{table}[!h]\n\\centering")
    latex_table = latex_table.replace("\\end{table}", caption_text + "\\end{table}")
    
    return latex_table

In [162]:
# funnel_10, many_well_32, student_t_mixture_50d, gaussian_mixture40_5d
tables = []
for filter_name in ["funnel_10", "many_well_32", "student_t_mixture_50d", "gaussian_mixture40_5d"]:
    print(f"Creating {filter_name} table")
    table = create_latex_table(filter_experiments(experiments, filter_name), filter_name)
    tables.append(table)

with open("tables.txt", "w") as f:
    f.write("\n".join(tables))

Creating funnel_10 table
Creating many_well_32 table
Creating student_t_mixture_50d table
Creating gaussian_mixture40_5d table
